# Taller Aplicado 2 — Plantilla base (trabajo autónomo)
**Benchmarking de modelos estadísticos clásicos y redes NAR/NARX sobre una serie real**

Esta plantilla trae resuelto el *andamiaje* del experimento: carga, validación del dataset, preprocesamiento sin fuga, diagnóstico, validación cruzada de origen móvil, métricas, residuos, prueba de Diebold-Mariano y exportación de resultados. **El diseño, la identificación y la optimización de los modelos son responsabilidad del equipo**: están marcados con `TODO-EQUIPO`.

El cuaderno corre de punta a punta desde el inicio con el modelo de referencia (ingenuo estacional). Los modelos que aún no implementen se omiten con un aviso.

## 0. Configuración del experimento

In [ ]:
# =====================================================================
#                       CONFIGURACIÓN (editar)
# =====================================================================
DATASET     = "appliances"   # 'appliances' | 'beijing' | 'jena' | 'propio' | 'sintetico' (prueba rápida)
RUTA_PROPIA = "/content/mis_datos.csv"   # Solo si DATASET='propio'. CSV con una columna de fecha.
COL_FECHA   = "fecha"                    # Nombre de la columna de fecha en el CSV propio.

OBJETIVO    = None    # None = valor por defecto del dataset (ver diccionario FUENTES).
EXOGENAS    = None    # Lista de columnas exógenas; None = valores por defecto; [] = sin exógenas.
FRECUENCIA  = None    # Remuestreo pandas ('h', 'D', ...). None = por defecto del dataset.
S           = None    # Periodo estacional en pasos de FRECUENCIA (24 para horario-diario, 7 para diario-semanal).

PROP_PRUEBA = 0.20    # Fracción final reservada como prueba EXTERNA (no se toca hasta el final).
N_PLIEGUES  = 5       # Orígenes de la validación cruzada de origen móvil dentro de la parte de desarrollo.
HORIZONTE   = 24      # Pasos a pronosticar desde cada origen.
VENTANA     = "expandible"   # 'expandible' o 'deslizante'
LONG_VENTANA_DESLIZANTE = 2000
SEMILLA     = 2026

## 1. Librerías

In [ ]:
import os, io, zipfile, urllib.request, random, time, itertools, warnings, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
warnings.filterwarnings("ignore")
random.seed(SEMILLA); np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": .3})

## 2. Carga de datos
Fuentes verificadas (UCI Machine Learning Repository y TensorFlow). Las descargas se guardan en `./datos/`.

In [ ]:
FUENTES = {
    "appliances": dict(url="https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip",
                       objetivo="Appliances", exogenas=["T_out", "RH_out", "Windspeed"], frecuencia="h", s=24),
    "beijing":    dict(url="https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip",
                       objetivo="PM2.5", exogenas=["TEMP", "DEWP", "PRES", "WSPM"], frecuencia="D", s=7,
                       estacion="Aotizhongxin"),
    "jena":       dict(url="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
                       objetivo="T (degC)", exogenas=["p (mbar)", "rh (%)"], frecuencia="h", s=24),
    "propio":     dict(objetivo=None, exogenas=[], frecuencia=None, s=None),
    "sintetico":  dict(objetivo="y", exogenas=["x"], frecuencia="D", s=7),
}

def descargar(url):
    os.makedirs("datos", exist_ok=True)
    destino = os.path.join("datos", url.split("/")[-1])
    if not os.path.exists(destino):
        print("Descargando", url)
        urllib.request.urlretrieve(url, destino)
    return destino

def leer_csv_de_zip(ruta_zip, contiene=".csv"):
    # Recorre zips anidados (Beijing trae un zip dentro de otro) hasta hallar el CSV pedido.
    with zipfile.ZipFile(ruta_zip) as z:
        for nombre in z.namelist():
            if nombre.endswith(".csv") and contiene in nombre:
                return pd.read_csv(z.open(nombre))
        for nombre in z.namelist():
            if nombre.endswith(".zip"):
                return leer_csv_de_zip(io.BytesIO(z.read(nombre)), contiene)
    raise FileNotFoundError(f"No se encontró un CSV que contenga '{contiene}'")

def cargar_datos(nombre):
    f = FUENTES[nombre]
    if nombre == "appliances":
        df = leer_csv_de_zip(descargar(f["url"]), "energydata")
        df.index = pd.to_datetime(df.pop("date"))
    elif nombre == "beijing":
        df = leer_csv_de_zip(descargar(f["url"]), f["estacion"])
        df.index = pd.to_datetime(df[["year", "month", "day", "hour"]])
        df = df.drop(columns=["No", "year", "month", "day", "hour", "wd", "station"])
    elif nombre == "jena":
        df = leer_csv_de_zip(descargar(f["url"]), "jena_climate")
        df.index = pd.to_datetime(df.pop("Date Time"), format="%d.%m.%Y %H:%M:%S")
        dup = df.index.duplicated()                       # el archivo trae marcas de tiempo repetidas
        print(f"Jena: {dup.sum()} registros duplicados eliminados (documentar en el informe)")
        df = df[~dup]
    elif nombre == "propio":
        df = pd.read_csv(RUTA_PROPIA)
        df.index = pd.to_datetime(df.pop(COL_FECHA))
    else:  # serie sintética con la misma lógica del Taller 1, para probar la plantilla sin internet
        t = np.arange(1500); rng = np.random.default_rng(SEMILLA)
        x = 18 + 6*np.sin(2*np.pi*(t-80)/365.25) + rng.normal(0, 1.5, t.size)
        y = 200 + .03*t + 10*np.sin(2*np.pi*t/7) + .35*(x-18)**2 + rng.normal(0, 3, t.size)
        df = pd.DataFrame({"y": y, "x": x}, index=pd.date_range("2020-01-01", periods=t.size, freq="D"))
    return df.sort_index()

cfg = FUENTES[DATASET]
OBJ  = OBJETIVO  or cfg["objetivo"]
EXO  = cfg["exogenas"] if EXOGENAS is None else EXOGENAS
FREC = FRECUENCIA or cfg["frecuencia"]
PER  = S or cfg["s"]
crudo = cargar_datos(DATASET)
print(crudo.shape); display(crudo.head())

## 3. Validación previa del dataset
Criterios mínimos (obligatorios para un **dataset propio** y recomendables para todos). El equipo debe reportar esta tabla en la sección de Metodología del informe.

In [ ]:
def validar_dataset(df, obj, exo, frec, per):
    serie = df[obj]
    paso = pd.Series(df.index).diff().dropna()
    criterios = {
        "C1 Índice temporal monótono y sin duplicados": df.index.is_monotonic_increasing and not df.index.duplicated().any(),
        "C2 Muestreo regular (paso modal >= 90% de los casos)": (paso == paso.mode()[0]).mean() >= 0.90,
        "C3 Faltantes en objetivo <= 10%": serie.isna().mean() <= 0.10,
        "C4 Longitud >= 10 periodos estacionales tras remuestreo": len(serie.resample(frec).mean().dropna()) >= 10*per if frec else True,
        "C5 Exógenas existen y son numéricas": all(c in df and pd.api.types.is_numeric_dtype(df[c]) for c in exo),
        "C6 Objetivo no constante": serie.std() > 0,
    }
    tabla = pd.DataFrame({"Cumple": criterios}); display(tabla)
    print(f"Paso modal original: {paso.mode()[0]} | faltantes objetivo: {100*serie.isna().mean():.2f}%")
    # C7 (manual): documentar origen, licencia, unidades y significado de cada variable.
    # C8 (manual): justificar que las exógenas estarían disponibles al momento de pronosticar.
    return tabla["Cumple"].all()

assert validar_dataset(crudo, OBJ, EXO, FREC, PER), "El dataset no cumple los criterios mínimos"

## 4. Preprocesamiento sin fuga de información
Regla de oro: **todo estadístico que transforme los datos (medias, desviaciones, tendencias, imputaciones con modelos) se estima solo con la parte de entrenamiento de cada pliegue.** Aquí solo se hacen transformaciones que no usan el futuro: remuestreo por agregación e imputación por interpolación temporal local.

In [ ]:
datos = crudo[[OBJ] + EXO].astype(float)
if FREC:
    datos = datos.resample(FREC).mean()                   # agregación causal dentro de cada intervalo
n_falt = datos.isna().sum()
datos = datos.interpolate(method="time", limit=6).ffill().bfill()   # huecos cortos; documentar en el informe
print("Faltantes imputados por columna:\n", n_falt)

N = len(datos)
n_dev = int((1 - PROP_PRUEBA) * N)                         # desarrollo = entrenamiento + validación cruzada
desarrollo, prueba = datos.iloc[:n_dev], datos.iloc[n_dev:]
print(f"Total {N} | desarrollo {len(desarrollo)} ({desarrollo.index[0]} a {desarrollo.index[-1]}) | "
      f"prueba externa {len(prueba)} ({prueba.index[0]} a {prueba.index[-1]})")

ax = datos[OBJ].plot(lw=.6, title=f"{DATASET}: {OBJ}")
ax.axvline(prueba.index[0], c="r", ls="--", label="inicio prueba externa"); ax.legend(); plt.show()

## 5. Diagnóstico estadístico (solo desarrollo)

In [ ]:
y_dev = desarrollo[OBJ]
for nombre, z in [("nivel", y_dev), (f"(1-B^{PER})", y_dev.diff(PER).dropna()),
                  (f"(1-B)(1-B^{PER})", y_dev.diff().diff(PER).dropna())]:
    print(f"{nombre:<20s} ADF p={adfuller(z, autolag='AIC')[1]:.4f} | KPSS p={kpss(z, nlags='auto')[1]:.4f}")

stl = STL(y_dev, period=PER, robust=True).fit()
stl.plot(); plt.gcf().set_size_inches(12, 7); plt.show()
F_T = max(0, 1 - np.var(stl.resid)/np.var(stl.trend + stl.resid))
F_S = max(0, 1 - np.var(stl.resid)/np.var(stl.seasonal + stl.resid))
print(f"F_T={F_T:.3f}  F_S={F_S:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(y_dev.diff(PER).dropna(), lags=4*PER, ax=ax[0]); plot_pacf(y_dev.diff(PER).dropna(), lags=4*PER, ax=ax[1], method="ywm")
plt.show()
if EXO:
    display(desarrollo.corr().round(3))

## 6. Validación cruzada de origen móvil
Para el pliegue $k$ con origen $o_k$: se entrena con $\{1,\dots,o_k-1\}$ (o una ventana deslizante) y se evalúa en $\{o_k,\dots,o_k+H-1\}$. Los orígenes se reparten en el último 40% del bloque de desarrollo. Así se respeta el orden temporal (Bergmeir & Benítez, 2012; Hyndman & Athanasopoulos, 2021, §5.10).

In [ ]:
def pliegues_origen_movil(n, n_pliegues, H, ventana="expandible", long_vent=2000, inicio=0.6):
    o_min, o_max = int(inicio*n), n - H
    origenes = np.linspace(o_min, o_max, n_pliegues).astype(int)
    for o in origenes:
        ini = 0 if ventana == "expandible" else max(0, o - long_vent)
        yield np.arange(ini, o), np.arange(o, o + H)

for k, (itr, ite) in enumerate(pliegues_origen_movil(len(desarrollo), N_PLIEGUES, HORIZONTE, VENTANA, LONG_VENTANA_DESLIZANTE)):
    print(f"Pliegue {k+1}: entrena {desarrollo.index[itr[0]]} a {desarrollo.index[itr[-1]]} "
          f"-> evalúa {desarrollo.index[ite[0]]} a {desarrollo.index[ite[-1]]}")

## 7. Modelos
**Contrato de todos los modelos:** `pronosticar(y_hist, X_hist, X_fut, H, **hp) -> np.ndarray de largo H`.
- `y_hist`: Serie (pandas) de entrenamiento del pliegue. `X_hist`: DataFrame de exógenas alineado (puede estar vacío).
- `X_fut`: exógenas del horizonte (supuesto de exógena conocida; si el equipo pronostica la exógena, debe hacerlo dentro de esta función).
- `hp`: hiperparámetros, para el análisis de sensibilidad.
- Dentro de la función se ajusta TODO lo dependiente de datos (escaladores, tendencias): así no hay fuga.

In [ ]:
def ingenuo_estacional(y_hist, X_hist, X_fut, H, **hp):
    # Referencia obligatoria: repite el último ciclo estacional observado.
    ult = y_hist.values[-PER:]
    return np.array([ult[h % PER] for h in range(H)])

def modelo_clasico(y_hist, X_hist, X_fut, H, **hp):
    # TODO-EQUIPO: implementar un modelo estadístico clásico (SARIMA/SARIMAX, ETS, TBATS, regresión dinámica).
    # Sugerencias:
    #   from statsmodels.tsa.statespace.sarimax import SARIMAX
    #   m = SARIMAX(y_hist, exog=X_hist if len(X_hist.columns) else None,
    #               order=hp.get("order", (1,0,1)), seasonal_order=hp.get("seasonal_order", (1,1,1,PER))).fit(disp=False)
    #   return m.forecast(H, exog=X_fut if len(X_fut.columns) else None).values
    # Justifique (p,d,q)(P,D,Q) con el diagnóstico de la sección 5 y con AIC/BIC, y valide residuos (Ljung-Box).
    raise NotImplementedError

def modelo_neuronal(y_hist, X_hist, X_fut, H, **hp):
    # TODO-EQUIPO: implementar NAR o NARX (u otra arquitectura justificada: MLP directo, LSTM, TCN...).
    # Pasos mínimos:
    #   1) Escalar y (y X) con estadísticos de y_hist/X_hist únicamente.
    #   2) Construir regresores con retardos: [y_{t-1..t-dy}, x_{t..t-dx}].
    #   3) Separar una cola de y_hist como validación para parada temprana.
    #   4) Entrenar en lazo abierto y pronosticar H pasos en LAZO CERRADO (recursivo) o con estrategia directa/MIMO.
    #   5) Desescalar y devolver un arreglo de largo H.
    raise NotImplementedError

MODELOS = {"Ingenuo estacional": (ingenuo_estacional, {}),
           "Clásico":            (modelo_clasico, {}),     # TODO-EQUIPO: hiperparámetros elegidos
           "Neuronal":           (modelo_neuronal, {})}

## 8. Motor de evaluación (no modificar)

In [ ]:
ESC_MASE = np.mean(np.abs(y_dev.values[PER:] - y_dev.values[:-PER]))   # escala MASE con desarrollo

def metricas(yr, yp):
    yr, yp = np.asarray(yr, float), np.asarray(yp, float); e = yr - yp
    nz = np.abs(yr) > 1e-8
    return {"RMSE": np.sqrt(np.mean(e**2)), "MAE": np.mean(np.abs(e)),
            "MAPE(%)": 100*np.mean(np.abs(e[nz]/yr[nz])) if nz.any() else np.nan,
            "sMAPE(%)": 200*np.mean(np.abs(e)/(np.abs(yr)+np.abs(yp)+1e-12)), "MASE": np.mean(np.abs(e))/ESC_MASE}

def evaluar(fun, hp, base, pliegues):
    reales, preds, tiempo = [], [], 0.0
    for itr, ite in pliegues:
        yh, Xh = base[OBJ].iloc[itr], base[EXO].iloc[itr]
        Xf = base[EXO].iloc[ite]
        t0 = time.time()
        p = np.asarray(fun(yh, Xh, Xf, len(ite), **hp), float).ravel()
        tiempo += time.time() - t0
        assert p.shape == (len(ite),), "El modelo debe devolver H valores"
        reales.append(base[OBJ].iloc[ite].values); preds.append(p)
    return np.array(reales), np.array(preds), tiempo

def correr_benchmark(base, pliegues, modelos):
    res, salidas = {}, {}
    for nombre, (fun, hp) in modelos.items():
        try:
            R, P, tt = evaluar(fun, hp, base, list(pliegues))
        except NotImplementedError:
            print(f"[aviso] '{nombre}' aún no implementado: se omite."); continue
        res[nombre] = {**metricas(R.ravel(), P.ravel()), "Tiempo(s)": tt}
        salidas[nombre] = (R, P)
    return pd.DataFrame(res).T, salidas

pl_cv = list(pliegues_origen_movil(len(desarrollo), N_PLIEGUES, HORIZONTE, VENTANA, LONG_VENTANA_DESLIZANTE))
tabla_cv, salidas_cv = correr_benchmark(desarrollo, pl_cv, MODELOS)
print(f"Validación cruzada de origen móvil ({N_PLIEGUES} pliegues, H={HORIZONTE})"); display(tabla_cv.round(4))

## 9. Análisis de sensibilidad de hiperparámetros
Defina una rejilla por modelo (mínimo 2 hiperparámetros × 3 niveles para el modelo neuronal). Se reporta media y desviación del RMSE por pliegue, y se recomienda repetir con ≥ 3 semillas para redes.

In [ ]:
REJILLAS = {
    # TODO-EQUIPO: ejemplo -> "Neuronal": {"dy": [6, 12, 24], "neuronas": [5, 10, 20]},
}

def sensibilidad(nombre, rejilla):
    fun, hp_base = MODELOS[nombre]
    claves = list(rejilla)
    filas = []
    for valores in itertools.product(*[rejilla[k] for k in claves]):
        hp = {**hp_base, **dict(zip(claves, valores))}
        R, P, tt = evaluar(fun, hp, desarrollo, pl_cv)
        rmse_pl = np.sqrt(np.mean((R - P)**2, axis=1))
        filas.append({**dict(zip(claves, valores)), "RMSE medio": rmse_pl.mean(), "RMSE desv": rmse_pl.std(), "Tiempo(s)": tt})
    t = pd.DataFrame(filas).sort_values("RMSE medio"); display(t.round(4))
    if len(claves) == 2:   # mapa de calor para dos hiperparámetros
        piv = t.pivot(index=claves[0], columns=claves[1], values="RMSE medio")
        plt.imshow(piv, cmap="viridis"); plt.colorbar(label="RMSE medio")
        plt.xticks(range(piv.shape[1]), piv.columns); plt.yticks(range(piv.shape[0]), piv.index)
        plt.xlabel(claves[1]); plt.ylabel(claves[0]); plt.title(f"Sensibilidad {nombre}"); plt.show()
    return t

tablas_sens = {m: sensibilidad(m, g) for m, g in REJILLAS.items()}

## 10. Evaluación final en la prueba externa (una sola vez, con los hiperparámetros elegidos)

In [ ]:
pl_test = list(pliegues_origen_movil(N, max(1, (len(prueba) - HORIZONTE)//HORIZONTE + 1), HORIZONTE,
                                     VENTANA, LONG_VENTANA_DESLIZANTE, inicio=n_dev/N))
tabla_test, salidas_test = correr_benchmark(datos, pl_test, MODELOS)
print(f"Prueba externa: {len(pl_test)} orígenes, H={HORIZONTE}"); display(tabla_test.round(4))

plt.figure()
for m, (R, P) in salidas_test.items():
    plt.plot(range(1, HORIZONTE+1), np.sqrt(np.mean((R-P)**2, axis=0)), marker="o", label=m)
plt.xlabel("horizonte h"); plt.ylabel("RMSE"); plt.title("RMSE por horizonte (prueba externa)"); plt.legend(); plt.show()

## 11. Residuos y significancia estadística

In [ ]:
def graficar_residuos(nombre, R, P):
    e_all = (R - P).ravel()
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.3))
    ax[0].plot(e_all, lw=.7); ax[0].set_title(f"{nombre}: residuos")
    ax[1].hist(e_all, bins=30); ax[1].set_title("Histograma")
    stats.probplot(e_all, dist="norm", plot=ax[2]); ax[2].set_title("Q-Q normal")
    plot_acf(e_all, lags=min(40, len(e_all)//3), ax=ax[3]); ax[3].set_title("ACF")
    plt.tight_layout(); plt.show()
    print(acorr_ljungbox(e_all, lags=[min(PER, len(e_all)//4)]).round(4))

def diebold_mariano(e1, e2, h=1):
    d = e1**2 - e2**2; T = len(d)
    g = [np.sum((d[k:]-d.mean())*(d[:T-k]-d.mean()))/T for k in range(h)]
    dm = d.mean()/np.sqrt((g[0] + 2*sum(g[1:]))/T) * np.sqrt((T+1-2*h+h*(h-1)/T)/T)
    return dm, 2*stats.t.sf(abs(dm), T-1)

for m, (R, P) in salidas_test.items():
    graficar_residuos(m, R, P)
nombres = list(salidas_test)
for a, b in itertools.combinations(nombres, 2):
    Ra, Pa = salidas_test[a]; Rb, Pb = salidas_test[b]
    dm, p = diebold_mariano((Ra-Pa).ravel(), (Rb-Pb).ravel(), h=HORIZONTE)
    print(f"DM {a} vs {b}: DM*={dm:.3f}, p={p:.4f}")

## 12. Exportación para el informe

In [ ]:
os.makedirs("resultados", exist_ok=True)
tabla_cv.to_csv("resultados/benchmark_cv.csv"); tabla_test.to_csv("resultados/benchmark_prueba.csv")
for m, t in tablas_sens.items():
    t.to_csv(f"resultados/sensibilidad_{m}.csv", index=False)
json.dump({"dataset": DATASET, "objetivo": OBJ, "exogenas": EXO, "frecuencia": FREC, "S": PER,
           "horizonte": HORIZONTE, "pliegues": N_PLIEGUES, "ventana": VENTANA,
           "hiperparametros": {m: hp for m, (_, hp) in MODELOS.items()}},
          open("resultados/configuracion.json", "w"), indent=2, default=str)
print("Archivos en ./resultados:", os.listdir("resultados"))